# D181 — MySQL Table Partitioning

Partitioning divides one logical table into smaller physical parts called **partitions**. Applications still query one table, while MySQL can access only the relevant partitions when a predicate allows **partition pruning**.

This notebook uses one table name, `sales_partitioned`, and the same dataset throughout. MySQL allows only one partitioning definition on a table, so each major section rebuilds that same table with a different strategy. Run the notebook from top to bottom.

## What partitioning is—and is not

Partitioning can help when a table is large and queries or maintenance naturally target a subset such as recent dates or one region. It can also make old-data removal fast with `DROP PARTITION`.

Partitioning is **not** a substitute for indexes. An index locates rows by key; partitioning first limits which physical partitions are considered. A poor partition key can add complexity without improving performance. Always verify pruning with `EXPLAIN` and measure real workloads.

### MySQL partitioning families covered

- `RANGE` and `RANGE COLUMNS`
- `LIST` and `LIST COLUMNS`
- `HASH` and `LINEAR HASH`
- `KEY` and `LINEAR KEY`
- composite partitioning using `RANGE` plus `HASH` or `KEY` subpartitions
- management with `ADD`, `REORGANIZE`, `DROP`, `TRUNCATE`, `COALESCE`, and maintenance operations

## 1. Connect to the running MySQL server

Defaults are `127.0.0.1:3306`, user `root`, and password `root`. Environment variables can override them. The notebook creates only the `partition_lab` database and its `sales_partitioned` table.

In [ ]:
import os
import mysql.connector
from mysql.connector import Error

MYSQL_CONFIG = {
    "host": os.environ.get("MYSQL_HOSTNAME", "127.0.0.1"),
    "port": int(os.environ.get("MYSQL_PORT", "3306")),
    "user": os.environ.get("MYSQL_USERNAME", "root"),
    "password": os.environ.get("MYSQL_PASSWORD", "root"),
}

server = mysql.connector.connect(**MYSQL_CONFIG)
cursor = server.cursor()
cursor.execute("CREATE DATABASE IF NOT EXISTS partition_lab")
cursor.close()
server.close()

connection = mysql.connector.connect(**MYSQL_CONFIG, database="partition_lab")
print("Connected:", connection.is_connected())
print("MySQL version:", connection.get_server_info())

## 2. Query helpers

These are the reliable D14/D16-style helpers. `execute_sql` fetches complete result sets, commits DDL/DML, and rolls back errors. `print_rows` formats returned rows.

In [ ]:
def print_rows(columns, rows):
    if not rows:
        print("No rows returned.")
        return
    text_rows = [["NULL" if value is None else str(value) for value in row]
                 for row in rows]
    widths = [len(str(column)) for column in columns]
    for row in text_rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]
    print(" | ".join(str(column).ljust(width)
                     for column, width in zip(columns, widths)))
    print("-+-".join("-" * width for width in widths))
    for row in text_rows:
        print(" | ".join(value.ljust(width)
                         for value, width in zip(row, widths)))


def execute_sql(sql, params=None):
    cursor = connection.cursor()
    try:
        cursor.execute(sql, params or ())
        if cursor.with_rows:
            columns = [item[0] for item in cursor.description]
            rows = cursor.fetchall()
            print_rows(columns, rows)
            return rows
        affected = cursor.rowcount
        connection.commit()
        print(f"Statement completed. Affected rows: {affected:,}")
        return affected
    except Error:
        connection.rollback()
        raise
    finally:
        cursor.close()

## 3. Reusable dataset and table loader

The dataset spans four years and four regions. `rebuild_table()` drops and recreates the same table name with the requested partition clause, then reloads identical rows. No second data table is needed.

The example intentionally has no primary key. MySQL requires **every unique key to include every column used in the partitioning expression**. Omitting a unique key lets us demonstrate several unrelated partition keys on the same table name. In production, design keys and partition keys together.

In [ ]:
sales_rows = [
    (1, 101, '2022-01-15', 'North', 'Online', 120.00, 'Completed'),
    (2, 102, '2022-05-10', 'South', 'Store',  250.00, 'Completed'),
    (3, 103, '2022-11-21', 'East',  'Online',  80.00, 'Cancelled'),
    (4, 104, '2023-02-03', 'West',  'Partner', 500.00, 'Completed'),
    (5, 101, '2023-06-18', 'North', 'Store',  310.00, 'Completed'),
    (6, 105, '2023-12-29', 'South', 'Online', 900.00, 'Returned'),
    (7, 106, '2024-01-09', 'East',  'Partner', 150.00, 'Completed'),
    (8, 107, '2024-04-22', 'West',  'Online', 1200.00, 'Completed'),
    (9, 108, '2024-08-14', 'North', 'Store',  450.00, 'Pending'),
    (10,109, '2024-12-31', 'South', 'Online', 700.00, 'Completed'),
    (11,110, '2025-01-01', 'East',  'Store',  200.00, 'Completed'),
    (12,111, '2025-03-17', 'West',  'Partner', 330.00, 'Pending'),
    (13,112, '2025-07-07', 'North', 'Online', 999.00, 'Completed'),
    (14,113, '2025-10-19', 'South', 'Store',   75.00, 'Cancelled'),
    (15,114, '2025-12-24', 'East',  'Online', 640.00, 'Completed'),
    (16,115, '2026-02-11', 'West',  'Store',  410.00, 'Completed'),
]

base_columns = """
    sale_id INT NOT NULL,
    customer_id INT NOT NULL,
    sale_date DATE NOT NULL,
    region VARCHAR(10) NOT NULL,
    channel VARCHAR(10) NOT NULL,
    amount DECIMAL(10,2) NOT NULL,
    status VARCHAR(12) NOT NULL
"""

insert_sql = """
INSERT INTO sales_partitioned
(sale_id, customer_id, sale_date, region, channel, amount, status)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

def rebuild_table(partition_clause):
    execute_sql("DROP TABLE IF EXISTS sales_partitioned")
    execute_sql(f"CREATE TABLE sales_partitioned ({base_columns}) {partition_clause}")
    cursor = connection.cursor()
    try:
        cursor.executemany(insert_sql, sales_rows)
        connection.commit()
        print(f"Loaded {len(sales_rows)} rows.")
    except Error:
        connection.rollback()
        raise
    finally:
        cursor.close()

## 4. Inspect partition metadata

`INFORMATION_SCHEMA.PARTITIONS` shows partition names, methods, boundary descriptions, estimated row counts, and storage. Row estimates can be approximate. The helper below is used after every rebuild.

In [ ]:
def show_partitions():
    return execute_sql("""
    SELECT partition_name, subpartition_name, partition_method,
           subpartition_method, partition_expression,
           partition_description, table_rows
    FROM information_schema.partitions
    WHERE table_schema = DATABASE()
      AND table_name = 'sales_partitioned'
    ORDER BY partition_ordinal_position, subpartition_ordinal_position
    """)

## 5. `RANGE`: expression-based intervals

`RANGE` assigns rows by increasing, non-overlapping upper bounds. `VALUES LESS THAN (2024)` excludes 2024; boundaries are always exclusive. `MAXVALUE` catches later values and prevents insert failures beyond the final explicit boundary.

Typical uses: years, months encoded as integers, monotonically increasing IDs, and retention windows.

In [ ]:
rebuild_table("""
PARTITION BY RANGE (YEAR(sale_date)) (
    PARTITION p_before_2023 VALUES LESS THAN (2023),
    PARTITION p_2023        VALUES LESS THAN (2024),
    PARTITION p_2024        VALUES LESS THAN (2025),
    PARTITION p_2025        VALUES LESS THAN (2026),
    PARTITION p_future      VALUES LESS THAN MAXVALUE
)
""")
show_partitions()

### `EXPLAIN` and partition pruning

In MySQL 8, ordinary `EXPLAIN` includes a `partitions` column. Some releases also accept the legacy spelling `EXPLAIN PARTITIONS`. The key evidence is the partition list: a constant date range should access only matching partitions. `EXPLAIN FORMAT=JSON` provides more detail.

In [ ]:
execute_sql("""
EXPLAIN
SELECT * FROM sales_partitioned
WHERE sale_date >= '2024-01-01' AND sale_date < '2025-01-01'
""")

execute_sql("""
EXPLAIN FORMAT=JSON
SELECT SUM(amount) FROM sales_partitioned
WHERE sale_date >= '2024-01-01' AND sale_date < '2025-01-01'
""")

### Pruning-friendly versus pruning-unfriendly predicates

Use constants and predicates that MySQL can relate to the partition expression. Functions, implicit conversions, or complex expressions can prevent pruning. Compare plans rather than assuming. The first query below should narrow partitions; the second asks MySQL to evaluate a function on every row and may access more partitions.

In [ ]:
execute_sql("EXPLAIN SELECT * FROM sales_partitioned WHERE sale_date = '2024-04-22'")
execute_sql("EXPLAIN SELECT * FROM sales_partitioned WHERE DATE_FORMAT(sale_date, '%Y') = '2024'")

### Explicit partition selection

`PARTITION (...)` restricts a statement to named partitions. This is useful for diagnostics and controlled maintenance, but hard-coding partition names in ordinary application queries couples code to physical design.

In [ ]:
execute_sql("""
SELECT sale_id, sale_date, amount
FROM sales_partitioned PARTITION (p_2024)
ORDER BY sale_date
""")

## 6. `RANGE COLUMNS`: direct date boundaries

`RANGE COLUMNS` compares column values directly and supports date, datetime, string, and tuple boundaries. It is often clearer than wrapping a date in `YEAR()`. Multiple columns are compared lexicographically from left to right.

In [ ]:
rebuild_table("""
PARTITION BY RANGE COLUMNS (sale_date) (
    PARTITION p_before_2024 VALUES LESS THAN ('2024-01-01'),
    PARTITION p_2024        VALUES LESS THAN ('2025-01-01'),
    PARTITION p_2025        VALUES LESS THAN ('2026-01-01'),
    PARTITION p_future      VALUES LESS THAN (MAXVALUE)
)
""")
show_partitions()
execute_sql("EXPLAIN SELECT * FROM sales_partitioned WHERE sale_date BETWEEN '2025-01-01' AND '2025-12-31'")

## 7. `LIST`: discrete integer expression values

`LIST` maps explicit integer results to partitions. Every inserted value must appear in one list; there is no `MAXVALUE` catch-all for `LIST`. Partition expressions accept only a restricted set of deterministic integer functions—an arbitrary `CASE` expression is not allowed. This example partitions the remainder from `MOD(customer_id, 4)`. Use `LIST COLUMNS` in the next section for direct string values such as regions.

In [ ]:
rebuild_table("""
PARTITION BY LIST (MOD(customer_id, 4)) (
    PARTITION p_remainder_0 VALUES IN (0),
    PARTITION p_remainder_1 VALUES IN (1),
    PARTITION p_remainder_2 VALUES IN (2),
    PARTITION p_remainder_3 VALUES IN (3)
)
""")
show_partitions()
execute_sql("EXPLAIN SELECT * FROM sales_partitioned WHERE customer_id = 109")

## 8. `LIST COLUMNS`: direct discrete values

`LIST COLUMNS` accepts non-integer columns and multiple-column tuples. It is the natural choice for a short, stable set of strings such as region codes. Here regions are paired into two business territories.

In [ ]:
rebuild_table("""
PARTITION BY LIST COLUMNS (region) (
    PARTITION p_north_south VALUES IN ('North', 'South'),
    PARTITION p_east_west   VALUES IN ('East', 'West')
)
""")
show_partitions()
execute_sql("EXPLAIN SELECT SUM(amount) FROM sales_partitioned WHERE region = 'East'")

## 9. `HASH`: even distribution from an integer expression

`HASH` uses MySQL's hashing rules to assign rows among a requested number of partitions. It is useful when there is no natural range/list grouping and balanced distribution matters. It is less suitable for dropping old date ranges because dates are scattered across partitions.

The expression must return an integer. Queries using equality on the hash key may prune; broad ranges commonly touch several or all partitions.

In [ ]:
rebuild_table("PARTITION BY HASH(customer_id) PARTITIONS 4")
show_partitions()
execute_sql("EXPLAIN SELECT * FROM sales_partitioned WHERE customer_id = 107")
execute_sql("EXPLAIN SELECT * FROM sales_partitioned WHERE customer_id BETWEEN 101 AND 110")

## 10. `LINEAR HASH`

`LINEAR HASH` uses a power-of-two algorithm. Adding, merging, splitting, and deleting partitions can move less data than regular hash partitioning, but distribution can be less even. Prefer regular `HASH` unless incremental partition changes are an important requirement.

In [ ]:
rebuild_table("PARTITION BY LINEAR HASH(customer_id) PARTITIONS 4")
show_partitions()
execute_sql("EXPLAIN SELECT * FROM sales_partitioned WHERE customer_id = 110")

## 11. `KEY`: MySQL chooses the hashing function

`KEY` resembles `HASH`, but MySQL supplies its internal hashing function and accepts one or more columns rather than an arbitrary expression. If the column list is omitted, MySQL uses the primary key or a suitable unique key; this lab names `sale_id` explicitly.

In [ ]:
rebuild_table("PARTITION BY KEY(sale_id) PARTITIONS 4")
show_partitions()
execute_sql("EXPLAIN SELECT * FROM sales_partitioned WHERE sale_id = 12")

## 12. `LINEAR KEY`

`LINEAR KEY` combines MySQL's key hashing with the linear power-of-two distribution algorithm. It has the same repartitioning trade-off as `LINEAR HASH`.

In [ ]:
rebuild_table("PARTITION BY LINEAR KEY(sale_id) PARTITIONS 4")
show_partitions()
execute_sql("EXPLAIN SELECT * FROM sales_partitioned WHERE sale_id = 8")

## 13. Composite partitioning with subpartitions

MySQL permits `HASH` or `KEY` subpartitioning beneath `RANGE` or `LIST` partitions. This can combine date-based lifecycle management with distribution inside each date partition. Every top-level partition must have the same number of subpartitions. More partitions are not automatically better; each carries metadata and operational cost.

In [ ]:
rebuild_table("""
PARTITION BY RANGE (YEAR(sale_date))
SUBPARTITION BY HASH(customer_id)
SUBPARTITIONS 2 (
    PARTITION p_before_2024 VALUES LESS THAN (2024),
    PARTITION p_2024        VALUES LESS THAN (2025),
    PARTITION p_2025        VALUES LESS THAN (2026),
    PARTITION p_future      VALUES LESS THAN MAXVALUE
)
""")
show_partitions()
execute_sql("""
EXPLAIN SELECT * FROM sales_partitioned
WHERE sale_date >= '2025-01-01' AND sale_date < '2026-01-01'
  AND customer_id = 112
""")

## 14. Partition management: split the catch-all partition

A `RANGE` table with `MAXVALUE` cannot receive a new partition after that catch-all with plain `ADD PARTITION`. Use `REORGANIZE PARTITION` to split it. This is a common rolling-window operation.

In [ ]:
rebuild_table("""
PARTITION BY RANGE COLUMNS (sale_date) (
    PARTITION p_before_2025 VALUES LESS THAN ('2025-01-01'),
    PARTITION p_future VALUES LESS THAN (MAXVALUE)
)
""")

execute_sql("""
ALTER TABLE sales_partitioned
REORGANIZE PARTITION p_future INTO (
    PARTITION p_2025 VALUES LESS THAN ('2026-01-01'),
    PARTITION p_future VALUES LESS THAN (MAXVALUE)
)
""")
show_partitions()

## 15. `TRUNCATE PARTITION` versus `DROP PARTITION`

- `TRUNCATE PARTITION p` quickly removes rows but retains the empty partition definition.
- `DROP PARTITION p` removes both its rows and definition. For `RANGE`/`LIST`, this is a fast archival or retention operation, but the deleted rows are not recoverable without backup.

The next cells demonstrate these operations on disposable lab data, then rebuild the table before continuing.

In [ ]:
execute_sql("SELECT COUNT(*) AS rows_before FROM sales_partitioned PARTITION (p_2025)")
execute_sql("ALTER TABLE sales_partitioned TRUNCATE PARTITION p_2025")
execute_sql("SELECT COUNT(*) AS rows_after FROM sales_partitioned PARTITION (p_2025)")
show_partitions()

In [ ]:
# Reload the common dataset before demonstrating DROP PARTITION.
rebuild_table("""
PARTITION BY RANGE COLUMNS (sale_date) (
    PARTITION p_before_2024 VALUES LESS THAN ('2024-01-01'),
    PARTITION p_2024 VALUES LESS THAN ('2025-01-01'),
    PARTITION p_2025 VALUES LESS THAN ('2026-01-01'),
    PARTITION p_future VALUES LESS THAN (MAXVALUE)
)
""")
execute_sql("ALTER TABLE sales_partitioned DROP PARTITION p_before_2024")
execute_sql("SELECT MIN(sale_date), COUNT(*) FROM sales_partitioned")
show_partitions()

## 16. Hash partition count management

For regular `HASH` or `KEY`, `ADD PARTITION PARTITIONS n` increases the count and redistributes rows. `COALESCE PARTITION n` reduces it and also redistributes rows. This can be expensive on large tables. Linear variants are designed to move less data during such changes.

In [ ]:
rebuild_table("PARTITION BY HASH(customer_id) PARTITIONS 4")
execute_sql("ALTER TABLE sales_partitioned ADD PARTITION PARTITIONS 2")
show_partitions()
execute_sql("ALTER TABLE sales_partitioned COALESCE PARTITION 2")
show_partitions()

## 17. Partition maintenance operations

MySQL supports `ANALYZE PARTITION`, `CHECK PARTITION`, `OPTIMIZE PARTITION`, `REBUILD PARTITION`, and `REPAIR PARTITION` where supported by the storage engine. `ANALYZE` refreshes optimizer statistics; `CHECK` verifies data and links; `REBUILD` recreates partition data. `OPTIMIZE` may rebuild/analyze the entire table for InnoDB and can be expensive. Use maintenance operations only after understanding locking, runtime, and engine behavior.

In [ ]:
execute_sql("ALTER TABLE sales_partitioned ANALYZE PARTITION ALL")
execute_sql("ALTER TABLE sales_partitioned CHECK PARTITION ALL")

## 18. Remove partitioning without deleting rows

`ALTER TABLE ... REMOVE PARTITIONING` converts the table to an ordinary nonpartitioned table while preserving rows. This is different from `DROP PARTITION`, which deletes the rows in the dropped partition.

In [ ]:
execute_sql("ALTER TABLE sales_partitioned REMOVE PARTITIONING")
show_partitions()
execute_sql("SELECT COUNT(*) AS rows_preserved FROM sales_partitioned")

## 19. Important design rules and limitations

1. Choose a partition key that matches frequent filters and lifecycle operations.
2. Verify pruning in the `partitions` column of `EXPLAIN`; do not assume it.
3. Every unique key must contain all columns used by the partitioning expression.
4. A table has one top-level partitioning scheme at a time.
5. `RANGE` boundaries are exclusive; provide a deliberate final boundary or `MAXVALUE`.
6. Every inserted `LIST` value must have a destination partition.
7. `HASH`/`KEY` improve distribution but do not naturally support time-based retention.
8. Partition count should be justified. Too many partitions increase metadata, planning, and maintenance costs.
9. Rows cannot reference different storage engines across partitions; a partitioned table uses one engine.
10. Foreign-key and partitioning support depends on MySQL version and engine restrictions; confirm the current manual before production design.
11. Partitioning does not automatically parallelize a query and does not guarantee speed.
12. Functions or type conversions in predicates can prevent pruning.
13. Global uniqueness is constrained by the unique-key rule; test key design early.
14. `DROP PARTITION` and `TRUNCATE PARTITION` delete data—back up and verify the exact target first.

## 20. Choosing a strategy

| Requirement | Likely strategy |
|---|---|
| Drop old months/years efficiently | `RANGE COLUMNS(date)` |
| Fixed territories or categories | `LIST COLUMNS(code)` |
| Balance rows without natural ranges | `HASH(integer_expression)` |
| Let MySQL hash one or more columns | `KEY(columns)` |
| Frequent hash partition-count changes | `LINEAR HASH` / `LINEAR KEY` |
| Date lifecycle plus distribution inside each date | `RANGE` + `HASH`/`KEY` subpartitions |

Start with workload evidence: table size, common predicates, retention rules, skew, indexes, and `EXPLAIN` output.

## 21. Practice

1. Rebuild the table using quarterly `RANGE COLUMNS` partitions for 2025.
2. Use `EXPLAIN` to prove that a March 2025 query accesses one quarter.
3. Add a 2026 partition by reorganizing `p_future`.
4. Partition by `LIST COLUMNS(channel)` and check pruning for `channel = 'Online'`.
5. Compare `HASH(customer_id)` plans for equality and range predicates.
6. Design a composite strategy for yearly retention and customer distribution.

## 22. Close the connection

In [ ]:
if connection.is_connected():
    connection.close()
print("MySQL connection closed.")